# 05 - Explainability and Anomaly Engine

Goal: explain model recommendations and identify unusual market events such as return shocks and volume spikes.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(ROOT / 'src'))

import joblib
import plotly.express as px

from invest_intel.data import load_processed
from invest_intel.explainability import detect_market_anomalies, feature_importance_frame

In [2]:
features = load_processed('nifty50_features.csv')
anomalies = detect_market_anomalies(features)
anomalies.tail(20)

,date,symbol,close,daily_return,return_z_60,volume,volume_zscore_20,is_return_shock,is_volume_spike
224567,2021-04-12,VEDL,213.20,-0.078453,-3.521310,14765942,-0.683775,True,False
233383,2021-04-12,ZEEL,181.10,-0.121087,-3.414405,24875466,2.438928,True,False
51220,2021-04-13,DRREDDY,4777.30,-0.043957,-2.104718,3937234,3.062244,False,True
143320,2021-04-13,M&M,811.25,0.079508,3.029892,12501628,3.464496,True,True
201352,2021-04-13,TCS,3104.05,-0.043893,-2.733709,8654596,3.194773,False,True
56522,2021-04-15,EICHERMOT,2412.60,-0.032580,-1.601414,1979939,3.364945,False,True
229876,2021-04-16,WIPRO,469.20,0.089389,4.037131,109361636,4.123800,True,True
77422,2021-04-20,HDFC,2415.90,-0.030674,-1.245065,7365765,3.042324,False,True
216891,2021-04-20,ULTRACEMCO,6200.85,-0.047452,-2.105423,1272134,3.273931,False,True
216892,2021-04-22,ULTRACEMCO,6092.30,-0.017506,-0.814676,1795988,3.370671,False,True


In [3]:
recent = anomalies[anomalies['date'] >= '2020-01-01']
px.scatter(recent, x='date', y='daily_return', color='symbol', size=recent['volume_zscore_20'].abs(), title='Recent return shocks and volume spikes')

In [4]:
model_path = ROOT / 'models' / 'stock_return_model.joblib'
if model_path.exists():
    artifact = joblib.load(model_path)
    importance = feature_importance_frame(artifact['model'], artifact['feature_cols'])
    display(importance.head(25))
    fig = px.bar(importance.head(20), x='importance', y='feature', orientation='h', title='Top model drivers')
    fig.show()
else:
    print('Run notebook 02 first to train and save a model artifact.')

ValueError: Estimator HistGradientBoostingRegressor does not expose native feature importance. Use permutation_importance_frame or shap_feature_importance_frame instead.